# 02 – RQ2: Predicting High-Severity Incident Outcomes

**Research Question:** Can risk domain, deployer type, and intent predict whether an AI incident results in a high-severity outcome?

**Hypothesis:** Enterprise-deployed incidents with unintentional intent in the AI system safety domain carry a significantly higher probability of high-severity outcomes.

**Methods:**
- Binary Logistic Regression (primary, interpretable)
- Random Forest classifier (validation, captures non-linear interactions)
- Evaluation: Accuracy, Precision, Recall, F1-score, AUC-ROC
- Train/test split: 80/20 stratified


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

FIGURES_DIR = Path('../outputs/figures')
df = pd.read_pickle('../outputs/incidents_prepared.pkl')

In [ ]:
# Identify relevant column names
domain_col  = [c for c in df.columns if 'risk domain'  in c.lower()][0]
intent_col  = [c for c in df.columns if 'intent'       in c.lower()][0]
entity_col  = [c for c in df.columns if 'responsible'  in c.lower()][0]

print('Columns identified:')
print(f'  Risk Domain:        {domain_col}')
print(f'  Intent:             {intent_col}')
print(f'  Responsible Entity: {entity_col}')

In [ ]:
# Define the binary outcome variable: High-Severity Incident
# Operationalised as incidents in the 'Lack of capability or robustness' subdomain
# OR where documented injuries / lives lost are > 0

subdomain_col = [c for c in df.columns if 'subdomain' in c.lower()]
print('Subdomain column candidates:', subdomain_col)

# Inspect unique values to confirm label
if subdomain_col:
    print(df[subdomain_col[0]].value_counts().head(10))

In [ ]:
# Build outcome variable: 1 = high-severity, 0 = not high-severity
# Adjust the string match below if the actual subdomain label differs in your dataset
HIGH_SEVERITY_SUBDOMAINS = [
    'lack of capability or robustness',
    'physical harm',
    'harm to physical safety',
]

if subdomain_col:
    sd = df[subdomain_col[0]].str.lower().fillna('')
    df['High_Severity'] = sd.apply(
        lambda x: 1 if any(s in x for s in HIGH_SEVERITY_SUBDOMAINS) else 0
    )
else:
    # Fallback: use Risk Domain containing 'safety'
    df['High_Severity'] = df[domain_col].str.lower().str.contains('safety', na=False).astype(int)

print('Outcome distribution:')
print(df['High_Severity'].value_counts())
print(f'Base rate: {df["High_Severity"].mean():.2%}')

In [ ]:
# Select and prepare features
features = [domain_col, intent_col, entity_col, 'Deployer_Type']
rq2_df = df[features + ['High_Severity']].dropna()
print(f'Records for RQ2 after dropping rows with missing features: {len(rq2_df):,}')

# One-hot encode categorical predictors
X = pd.get_dummies(rq2_df[features], drop_first=False)
y = rq2_df['High_Severity']

print(f'Feature matrix shape: {X.shape}')

In [ ]:
# Train / test split (80/20, stratified on outcome)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows')

In [ ]:
# --- Model 1: Binary Logistic Regression ---
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
y_pred_lr   = lr.predict(X_test)
y_prob_lr   = lr.predict_proba(X_test)[:, 1]

print('--- Logistic Regression ---')
print(classification_report(y_test, y_pred_lr, target_names=['Not High-Severity', 'High-Severity']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')

In [ ]:
# Odds ratios from logistic regression
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_[0],
    'Odds Ratio': np.exp(lr.coef_[0])
}).sort_values('Odds Ratio', ascending=False)

print('Top predictors by Odds Ratio:')
print(coef_df.head(10).to_string(index=False))

In [ ]:
# --- Model 2: Random Forest (validation) ---
rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf   = rf.predict(X_test)
y_prob_rf   = rf.predict_proba(X_test)[:, 1]

print('--- Random Forest ---')
print(classification_report(y_test, y_pred_rf, target_names=['Not High-Severity', 'High-Severity']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}')

In [ ]:
# Feature importance from Random Forest
fi_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
fi_df.head(15).plot(kind='barh', x='Feature', y='Importance', ax=ax, legend=False)
ax.set_title('Random Forest Feature Importances (Top 15)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq2_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# AUC-ROC comparison plot
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})')
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest      (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves: Logistic Regression vs. Random Forest', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'rq2_roc_curves.png', dpi=150)
plt.show()